# 01 — First GPU Slurm job: Aer on an A100/H200

**Level:** beginner → intermediate. **Environment:** CPU Jupyter workbench + approved Aer-GPU Apptainer SIF. **GPU:** one scheduler-allocated device. **Credentials:** your authenticated POSIX/Slurm principal; never shared root/sudo credentials. **Expected time:** queue-dependent, requested walltime 02:00:00. **Artifacts:** `results/aer-gpu-JOBID.json` on submission filesystem, scheduler logs/accounting.

**This notebook is not an interactive GPU session.** It requires native `sbatch` to be available under the researcher's own effective UID on an approved submit host. The existing Kubernetes notebook does **not** yet have that secure identity path activated. Do not add a privileged runuser helper.

For Axis H200, `SLURM_SUBMIT_DIR` and outputs must be staged to/from the remote site separately: a mounted ACE home or SSHFS symlink is **not** assumed. See `docs/M5-BATCH-FIRST-SIMULATOR-PATH.md`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

ROOT = next((d for d in (Path.cwd(), *Path.cwd().parents) if (d / "slurm/aer-gpu.sbatch").is_file()), None)
assert ROOT is not None, "Launch from a checkout containing slurm/aer-gpu.sbatch"
assert shutil.which("sbatch"), "Native sbatch unavailable: approved per-user Slurm submission is not configured"
PARTITION = os.getenv("SLURM_GPU_PARTITION")
IMAGE = os.getenv("QISKIT_AER_GPU_IMAGE")
assert PARTITION and IMAGE, "Site must publish an approved GPU partition and image path"
assert Path(IMAGE).is_absolute(), "Use an approved absolute, site-visible image path"
assert "," not in IMAGE and "," not in PARTITION, "Invalid Slurm export/partition value"
print("Submission ready:", PARTITION, IMAGE)

## Submit once (one GPU)

This is an explicit user-triggered action. Re-running this cell submits **another** job. Do not retry blindly after a network failure: inspect your queue first. Never run as the portal's service account.

In [ ]:
command = [
    "sbatch", "--parsable", f"--partition={PARTITION}",
    f"--export=ALL,QISKIT_AER_GPU_IMAGE={IMAGE},AER_QUBITS=22,AER_SHOTS=2048",
    "slurm/aer-gpu.sbatch",
]
response = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
JOB_ID = response.stdout.strip().split(";", 1)[0]
assert JOB_ID.isdigit(), f"Unexpected Slurm job ID: {response.stdout!r}"
import json
STATE_DIR = Path.home() / ".local" / "state" / "quantum-workflows"
STATE_DIR.mkdir(parents=True, exist_ok=True, mode=0o700)
STATE_FILE = STATE_DIR / f"aer-gpu-{JOB_ID}.json"
STATE_FILE.write_text(json.dumps({
    "job_id": JOB_ID,
    "site": os.getenv("SLURM_EXECUTION_SITE", "site-not-recorded"),
    "partition": PARTITION,
}, indent=2) + "\\n")
STATE_FILE.chmod(0o600)
print(f"Submitted durable Slurm job {JOB_ID}; notebook may close now.")
print("Saved private job reference:", STATE_FILE)

## Observe without reserving the GPU yourself

The Slurm job survives notebook/browser closure. On restart, restore your own job ID after authenticating; do not scan other users' records.

In [ ]:
import json
import subprocess
from pathlib import Path
STATE_DIR = Path.home() / ".local" / "state" / "quantum-workflows"
if "JOB_ID" not in globals():
    saved_jobs = sorted(
        STATE_DIR.glob("aer-gpu-*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    ) if STATE_DIR.exists() else []
    assert saved_jobs, "No recorded job reference. Restore your own job ID explicitly."
    state = json.loads(saved_jobs[0].read_text())
    JOB_ID = str(state["job_id"])
    print("Restored", JOB_ID, "recorded for", state.get("site"), state.get("partition"))
    print("Check that you are connected to the same site before querying.")
assert str(JOB_ID).isdigit()
for args in (["squeue", "-h", "-j", str(JOB_ID), "-o", "%i|%T|%M|%R"],
             ["sacct", "-nP", "-X", "-j", str(JOB_ID), "--format=JobID,State,ExitCode,Elapsed,AllocTRES"]):
    proc = subprocess.run(args, text=True, capture_output=True, check=False)
    print(args[0], proc.stdout.strip() or proc.stderr.strip())

## Verify scientific output, not just a COMPLETED state

Only local/shared-filesystem mode is implemented by this reference lab. The H200 connector must explicitly stage outputs back and validate checksum, ownership and manifest; it must not silently attempt a cross-site NFS or SSHFS mount.

In [ ]:
import json
assert "JOB_ID" in globals()
if "ROOT" not in globals():
    ROOT = next((d for d in (Path.cwd(), *Path.cwd().parents)
                 if (d / "slurm/aer-gpu.sbatch").is_file()), None)
assert ROOT is not None, "Open the notebook from the source checkout to find local results"
output = ROOT / "results" / f"aer-gpu-{JOB_ID}.json"
if output.is_file():
    data = json.loads(output.read_text())
    assert data["scientific_check_passed"], "Scheduler completed but numerical validation failed"
    print({key: data.get(key) for key in ("slurm_job_id","qubits","shots","duration_seconds","qiskit_aer_version","scientific_check_passed")})
else:
    print("Not yet staged:", output, "Check sacct and authorized site stage-out.")

**Cleanup:** after examining the output, remove only your own disposable job files according to the facility retention policy. To cancel **your own** running job, use native `scancel JOB_ID` under your own Slurm identity; the notebook does not cancel automatically.